In [ ]:
import scanpy as sc
adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-05-07-01/data.h5ad")


In [ ]:
adata_raw = adata.copy()

In [ ]:
import sys
sys.path.append("../../")
from src.training import helpers as tr_h

mask_genes = tr_h.get_top_k_most_present_genes(
    adata_raw, k=3501
)

# mask the genes
adata = adata_raw[:, mask_genes]


In [ ]:
import numpy as np
# mask samples
# non_nan_percentage = np.sum(~np.isnan(adata.X), axis=1) / adata.X.shape[1]
non_zero_non_nan_mask = ~np.isnan(adata.X) & ~(adata.X == 0)

non_zero_non_nan_mask_pct = np.sum(non_zero_non_nan_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples = non_zero_non_nan_mask_pct >= 0.3
print(
    f"Filtering out {np.sum(~mask_samples)} / {len(mask_samples)} samples with less than 30% non-NaN values"
)

print(f"adata shape: {adata.shape}")
# apply the mask to the AnnData object
adata = adata[mask_samples, :]
print(f"adata shape: {adata.shape}")


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib import cm

def fill_nans(X):
    X[np.isnan(adata.X)] = 0
    return X
def plot_tsne(adata:sc.AnnData, label:str)->None:
    
    # get data
    X = adata.X.copy()
    lab = adata.obs[label].values
    codes = lab.cat.codes.to_numpy()          # 0..K-1
    cats  = lab.cat.categories.to_list()      # ['Microarray','RNA-Seq', ...]

    plt.figure(figsize=(8,6))
    cmap = cm.get_cmap("tab10", len(cats))    # discrete colors
    sc = plt.scatter(X_embedded[:,0], X_embedded[:,1],
                     c=codes, cmap=cmap, s=8, alpha=0.7)

    cbar = plt.colorbar(sc, ticks=range(len(cats)))
    cbar.ax.set_yticklabels(cats)
    plt.title(f"t-SNE colored by {label}")
    plt.xlabel("t-SNE 1"); plt.ylabel("t-SNE 2")
    plt.tight_layout()
    plt.show()
plot_tsne(adata, "library")


Traceback (most recent call last):
Exception ignored in: 'zmq.backend.cython._zmq.Frame.__del__'
Traceback (most recent call last):
  File "_zmq.py", line 141, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt: 
  File "/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3577, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2238562/2397655042.py", line 27, in <module>
    plot_tsne(adata, "library")
  File "/tmp/ipykernel_2238562/2397655042.py", line 13, in plot_tsne
    codes = lab.cat.codes.to_numpy()          # 0..K-1
            ^^^^^^^
AttributeError: 'Categorical' object has no attribute 'cat'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "_zmq.py", line 141, in zmq.backend.cython._zmq._check_rc
KeyboardInterrupt


AttributeError: 'Categorical' object has no attribute 'cat'

In [ ]:
X = adata.X

In [ ]:
np.isnan(X).sum()